# 🫁 AI for Health — Sleep Apnea Detection
**SRIP 2026 | Health Sensing Assignment**

This notebook implements the complete pipeline for detecting breathing irregularities during sleep.
All stable utility code lives in `.py` scripts and is imported here. The main pipeline logic
lives directly in this notebook, so it can be edited in Colab without needing to push to Git.

---
| Section | Description |
|---|---|
| 0 | Environment Setup |
| 1 | Data Setup & Extraction |
| 2 | Signal Visualization |
| 3 | Dataset Creation |
| 4 | Model Architecture |
| 5 | LOPO Cross-Validation Training |
| 6 | Results & Analysis |
| 7 | Export & Download |

---
## Section 0 — Environment Setup
Clone the repo, install requirements, and register the project root on the Python path so all `.py` modules can be imported directly.

In [ ]:
!git clone https://github.com/dmist08/AI-For-Health.git
!pip install -q -r AI-For-Health/requirements.txt

import sys
REPO = '/content/AI-For-Health'
sys.path.insert(0, REPO)

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## Section 1 — Data Setup
Upload your `Data.zip` to Colab, extract it, then run `setup_data.py` to rename files into the standardized `Data/AP01..AP05` structure.

In [ ]:
import zipfile, os

ZIP_PATH  = '/content/Data.zip'   # Change if your zip has a different name
DEST_PATH = f'{REPO}/internship/'

os.makedirs(DEST_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(DEST_PATH)
print('Extracted to:', os.listdir(DEST_PATH))

In [ ]:
# Standardize file and folder naming across participants
import subprocess, sys
result = subprocess.run([sys.executable, f'{REPO}/setup_data.py'], cwd=REPO, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0: print('ERROR:', result.stderr)

---
## Section 2 — Signal Visualization
Generate a 4-panel figure (Nasal Airflow, Thoracic Movement, SpO₂, Sleep Stages) for each participant.
Breathing events are overlaid as coloured spans. PDFs saved to `Visualizations/`.

In [ ]:
import subprocess, sys, os

DATA_DIR = f'{REPO}/Data'
os.makedirs(f'{REPO}/Visualizations', exist_ok=True)

for p in ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']:
    print(f'Plotting {p} ...')
    result = subprocess.run(
        [sys.executable, f'{REPO}/scripts/vis.py', '-name', f'{DATA_DIR}/{p}'],
        cwd=REPO, capture_output=True, text=True
    )
    print(result.stdout.strip() or f'{p} done')
    if result.returncode != 0: print('ERROR:', result.stderr)

print('\nVisualization PDFs saved to Visualizations/')

---
## Section 3 — Dataset Creation

**Processing pipeline:**
1. Bandpass filter Nasal Airflow + Thoracic at **0.17–0.40 Hz** (human breathing range)
2. Lowpass filter SpO₂ at **1.0 Hz** (removes sensor noise, keeps desaturation trends)
3. Upsample SpO₂ from **4 Hz → 32 Hz** (so all signals have equal length)
4. Stack 3 channels into shape **(3, 960)** per 30-second window
5. Slide window with **50% overlap** (15-second step)
6. Label each window: **0 = Normal**, **1 = Event** (Hypopnea or any Apnea variant)

Edit any parameters in the cells below directly in Colab if needed.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from scipy.signal import butter, filtfilt, resample_poly

# Import stable utility modules from the .py scripts
from scripts.parsers import parse_signal_file, parse_events_file
from src.config import config
from src.utils import setup_logger

logger = setup_logger('Dataset')

In [ ]:
# ─── Signal Processing Helpers ────────────────────────────────────────────────

def apply_bandpass(signal, fs, lo=0.17, hi=0.40, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lo/nyq, hi/nyq], btype='bandpass')
    return filtfilt(b, a, signal)

def apply_lowpass(signal, fs, cutoff=1.0, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff/nyq, btype='low')
    return filtfilt(b, a, signal)

def upsample_spo2(signal, from_fs=4, to_fs=32):
    return resample_poly(signal, up=to_fs, down=from_fs)

def label_to_int(label):
    return 0 if label.strip() == 'Normal' else 1

def get_window_label(win_start, win_sec, df_events):
    if df_events.empty:
        return 'Normal'
    win_end   = win_start + win_sec
    threshold = win_sec * 0.5
    for _, ev in df_events.iterrows():
        ev_end  = ev['start_sec'] + ev['duration_sec']
        overlap = max(0, min(win_end, ev_end) - max(win_start, ev['start_sec']))
        if overlap > threshold:
            return str(ev['label']).strip()
    return 'Normal'

def extract_windows(signal, fs, window_sec, step_sec):
    win_pts, step_pts = window_sec * fs, step_sec * fs
    wins, starts = [], []
    for i in range(0, len(signal) - win_pts + 1, step_pts):
        wins.append(signal[i:i+win_pts])
        starts.append(i / fs)
    return wins, starts

In [ ]:
# ─── Per-Participant Processing ────────────────────────────────────────────────

def process_participant(p_dir):
    p_id = Path(p_dir).name
    try:
        flow,   rec_start = parse_signal_file(str(Path(p_dir)/'nasal_airflow.txt'))
        thorac, _         = parse_signal_file(str(Path(p_dir)/'thoracic_movement.txt'))
        spo2,   _         = parse_signal_file(str(Path(p_dir)/'spo2.txt'))
    except Exception as e:
        logger.error(f'{p_id}: Signal load failed — {e}')
        return None

    events_path = Path(p_dir) / 'flow_events.txt'
    df_events = parse_events_file(str(events_path), rec_start) if events_path.exists() \
                else pd.DataFrame()
    if df_events.empty:
        logger.warning(f'{p_id}: No events — all windows will be Normal')

    flow_f   = apply_bandpass(flow.values,   config.FS_FLOW)
    thorac_f = apply_bandpass(thorac.values, config.FS_FLOW)
    spo2_f   = apply_lowpass(spo2.values,   config.FS_SPO2)
    spo2_up  = upsample_spo2(spo2_f, from_fs=config.FS_SPO2, to_fs=config.FS_FLOW)

    # Align to same length
    n = min(len(flow_f), len(thorac_f), len(spo2_up))
    flow_f, thorac_f, spo2_up = flow_f[:n], thorac_f[:n], spo2_up[:n]

    f_wins,  starts = extract_windows(flow_f,   config.FS_FLOW, config.WINDOW_SEC, config.STEP_SEC)
    t_wins,  _      = extract_windows(thorac_f, config.FS_FLOW, config.WINDOW_SEC, config.STEP_SEC)
    s_wins,  _      = extract_windows(spo2_up,  config.FS_FLOW, config.WINDOW_SEC, config.STEP_SEC)
    n_wins = min(len(f_wins), len(t_wins), len(s_wins))

    X, y, label_strs = [], [], []
    for i in range(n_wins):
        win   = np.stack([f_wins[i], t_wins[i], s_wins[i]], axis=0)  # (3, 960)
        lbl   = get_window_label(starts[i], config.WINDOW_SEC, df_events)
        X.append(win)
        label_strs.append(lbl)
        y.append(label_to_int(lbl))

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    logger.info(f'{p_id}: {len(X)} windows | {dict(Counter(label_strs))}')
    return {'X': X, 'y': y}


# ─── Run across all participants ───────────────────────────────────────────────

DATA_DIR    = f'{REPO}/Data'
DATASET_DIR = f'{REPO}/Dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

dataset = {}
for p_dir in tqdm(sorted(Path(DATA_DIR).iterdir()), desc='Processing'):
    result = process_participant(p_dir)
    if result:
        dataset[p_dir.name] = result

with open(f'{DATASET_DIR}/breathing_dataset.pkl', 'wb') as f:
    pickle.dump(dataset, f)

print('\nDataset saved!')

In [ ]:
# Inspect label distribution
print(f'{"Participant":<12} {"Total":>8} {"Normal (0)":>12} {"Event (1)":>12}')
print('-' * 46)
for p, d in sorted(dataset.items()):
    c = Counter(d['y'].tolist())
    print(f'{p:<12} {len(d["y"]):>8} {c.get(0,0):>12} {c.get(1,0):>12}')

---
## Section 4 — Model Architecture

**SimpleCNN** — single-branch 1D CNN for binary event classification.

```
Input (Batch, 3, 960)
  → ConvBlock(3→32,  k=7, pool=4)  →  (Batch, 32,  240)
  → ConvBlock(32→64, k=5, pool=4)  →  (Batch, 64,   60)
  → ConvBlock(64→128,k=3, pool=4)  →  (Batch, 128,  15)
  → GlobalAvgPool                  →  (Batch, 128)
  → Dropout(0.5) → Linear(128→2)
```

In [ ]:
import torch
from models.cnn_model import SimpleCNN, count_parameters

model = SimpleCNN(num_classes=2)
print(f'Parameters: {count_parameters(model):,}')
dummy = torch.randn(4, 3, 960)
print(f'Input:  {dummy.shape}  →  Output: {model(dummy).shape}')

---
## Section 5 — LOPO Cross-Validation Training

**Leave-One-Participant-Out CV** (mandatory per assignment):
- 5 folds — each participant is the test set exactly once
- Normalization stats computed **only on the training fold** (prevents leakage)
- `CrossEntropyLoss` with class weights handles the Normal >> Event imbalance
- Early stopping monitors **validation loss** (patience = 15 epochs)
- Logs saved to `logs/`, confusion matrices to `results/`

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import set_seed, get_device
from src.config import config
from models.cnn_model import SimpleCNN

logger_train = setup_logger('Training')
RESULTS_DIR  = f'{REPO}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
class BreathingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class EarlyStopping:
    def __init__(self, patience=15):
        self.patience, self.counter, self.best, self.stop = patience, 0, float('inf'), False
    def step(self, val_loss):
        if val_loss < self.best:
            self.best, self.counter = val_loss, 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            total_loss += criterion(logits, y).item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())
    return total_loss / len(loader), np.array(labels), np.array(preds)


def train_fold(fold, train_p, test_p, dataset, device):
    logger_train.info(f"\n{'='*50}\nFold {fold}/5 — Test: {test_p}\n{'='*50}")

    X_train = np.concatenate([dataset[p]['X'] for p in train_p])
    y_train = np.concatenate([dataset[p]['y'] for p in train_p])
    X_test, y_test = dataset[test_p]['X'], dataset[test_p]['y']

    # Normalize per channel using train stats only
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2),  keepdims=True) + 1e-8
    X_train = (X_train - mean) / std
    X_test  = (X_test  - mean) / std

    train_loader = DataLoader(BreathingDataset(X_train, y_train),
                              batch_size=config.BATCH_SIZE, shuffle=True,
                              num_workers=config.DATALOADER_WORKERS)
    test_loader  = DataLoader(BreathingDataset(X_test, y_test),
                              batch_size=config.BATCH_SIZE, shuffle=False,
                              num_workers=config.DATALOADER_WORKERS)

    model     = SimpleCNN(num_classes=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=0.01)

    counts  = Counter(y_train.tolist())
    total   = len(y_train)
    weights = torch.tensor([total / counts.get(i, 1) for i in range(2)], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    es = EarlyStopping(patience=15)
    best_val_loss, best_preds, best_labels = float('inf'), None, None

    for epoch in range(config.EPOCHS):
        model.train()
        train_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        val_loss, val_y, val_p = evaluate(model, test_loader, criterion, device)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            _, _, f1, _ = precision_recall_fscore_support(val_y, val_p, average='binary', zero_division=0)
            logger_train.info(f'Ep {epoch+1:3d} | TLoss {train_loss:.4f} | VLoss {val_loss:.4f} | F1 {f1:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds, best_labels = val_p.copy(), val_y.copy()

        es.step(val_loss)
        if es.stop:
            logger_train.info(f'Early stopping at epoch {epoch+1}')
            break

    return best_labels, best_preds

In [ ]:
set_seed(config.SEED)
device = get_device()
logger_train.info(f'Device: {device}')

participants  = sorted(dataset.keys())
all_y, all_p  = [], []
fold_results  = []

for i, test_p in enumerate(participants):
    train_p = [p for p in participants if p != test_p]
    y_true, y_pred = train_fold(i+1, train_p, test_p, dataset, device)

    all_y.extend(y_true)
    all_p.extend(y_pred)

    acc        = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    fold_results.append({'Fold': i+1, 'Test': test_p,
                         'Accuracy': round(acc,4), 'Precision': round(prec,4),
                         'Recall': round(rec,4), 'F1': round(f1,4)})

    # Per-fold confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Event'], yticklabels=['Normal','Event'])
    ax.set_title(f'Fold {i+1} — Test: {test_p}')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/cm_fold_{i+1}_{test_p}.png')
    plt.show()
    plt.close()

logger_train.info('\nLOPO CV COMPLETE')

---
## Section 6 — Results & Analysis

In [ ]:
import pandas as pd

df_results = pd.DataFrame(fold_results)
df_results.to_csv(f'{RESULTS_DIR}/lopo_metrics.csv', index=False)

display(df_results.style.format({'Accuracy':'.3f','Precision':'.3f','Recall':'.3f','F1':'.3f'})
              .set_caption('LOPO Cross-Validation Results'))

o_acc = accuracy_score(all_y, all_p)
o_prec, o_rec, o_f1, _ = precision_recall_fscore_support(all_y, all_p, average='binary', zero_division=0)
logger_train.info(f'Aggregate | Acc: {o_acc:.4f} | Prec: {o_prec:.4f} | Rec: {o_rec:.4f} | F1: {o_f1:.4f}')
print(f'Mean F1: {df_results["F1"].mean():.3f}   |   Aggregate F1: {o_f1:.3f}')

In [ ]:
# Aggregate confusion matrix
cm_all = confusion_matrix(all_y, all_p, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_all, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Normal','Event'], yticklabels=['Normal','Event'])
ax.set_title('Aggregate Confusion Matrix (All Folds)')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cm_aggregate.png')
plt.show()

---
## Section 7 — Export & Download
Zip all outputs (Dataset, Visualizations, Results, Logs) and download to your local machine.

In [ ]:
import zipfile
from google.colab import files

OUTPUT_ZIP = '/content/AI_for_Health_Outputs.zip'
FOLDERS    = ['Dataset', 'Visualizations', 'results', 'logs']

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in FOLDERS:
        fp = os.path.join(REPO, folder)
        if not os.path.exists(fp):
            print(f'Skipping {folder} (not found)')
            continue
        for root, _, fnames in os.walk(fp):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                zf.write(fpath, os.path.relpath(fpath, REPO))

print(f'Archive ready: {OUTPUT_ZIP}')
files.download(OUTPUT_ZIP)